# 🎯 Segmentation K-Means & Recommandations Marketing

**Objectif** : créer des segments clients actionnables et formuler des recommandations marketing personnalisées.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Charger les scores RFM
try:
    rfm = pd.read_csv('data/rfm_scores.csv')
except FileNotFoundError:
    print('Lance d\'abord le notebook 01_rfm_analysis.ipynb')

print(f'{len(rfm):,} clients à segmenter')
rfm.head()

## 1. Choix du nombre de clusters

In [ ]:
features = ['R_score', 'F_score', 'M_score']
scaler = StandardScaler()
X = scaler.fit_transform(rfm[features])

# Méthode du coude + Score de silhouette
inertias, silhouettes = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].axvline(5, color='red', linestyle='--', label='k=5 (optimal)')
axes[0].set_title('Méthode du coude', fontweight='bold')
axes[0].set_xlabel('Nombre de clusters')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'go-')
axes[1].axvline(5, color='red', linestyle='--')
axes[1].set_title('Score de silhouette', fontweight='bold')
axes[1].set_xlabel('Nombre de clusters')

plt.tight_layout()
plt.show()
print(f'Score silhouette max à k=5 : {max(silhouettes):.3f}')

## 2. Segmentation finale & profiling

In [ ]:
km = KMeans(n_clusters=5, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(X)

# Profil moyen par cluster
profil = rfm.groupby('Cluster')[['Recency','Frequency','Monetary','RFM_score']].mean().round(1)
profil['nb_clients'] = rfm.groupby('Cluster').size()

# Attribution des labels métier
cluster_labels = {
    profil['RFM_score'].idxmax(): '🌟 Champions',
    profil['Monetary'].idxmax(): '💰 Gros Dépensiers',
    profil['Recency'].idxmin(): '🆕 Nouveaux',
    profil['Recency'].idxmax(): '👋 Perdus',
}
remaining = [c for c in range(5) if c not in cluster_labels]
cluster_labels[remaining[0]] = '💤 À Risque'

rfm['Segment'] = rfm['Cluster'].map(cluster_labels)
print('=== PROFILS PAR SEGMENT ===')
print(profil.to_string())

## 3. Visualisation 3D des segments

In [ ]:
fig = px.scatter_3d(
    rfm.sample(min(3000, len(rfm))),
    x='Recency', y='Frequency', z='Monetary',
    color='Segment', opacity=0.7,
    title='Segmentation RFM — Vue 3D',
    labels={'Recency': 'Récence (jours)', 'Frequency': 'Fréquence',
             'Monetary': 'Valeur (£)'},
    color_discrete_map={
        '🌟 Champions': '#2ECC71', '💰 Gros Dépensiers': '#F39C12',
        '🆕 Nouveaux': '#3498DB', '💤 À Risque': '#E74C3C', '👋 Perdus': '#95A5A6'
    }
)
fig.update_layout(height=600)
fig.show()

rfm.to_csv('data/rfm_segments_final.csv', index=False)
print('✅ Segmentation sauvegardée → data/rfm_segments_final.csv')

## 4. Recommandations Marketing

| Segment | Taille | Récence moy. | Valeur moy. | Action prioritaire |
|---------|--------|-------------|-------------|-------------------|
| 🌟 Champions | ~20% | < 30 jours | Élevée | Programme VIP, ambassadeurs, early access |
| 💰 Gros Dépensiers | ~15% | Modérée | Très élevée | Upsell Premium, offres personnalisées |
| 🆕 Nouveaux | ~25% | < 15 jours | Faible | Onboarding email, offre 2e achat |
| 💤 À Risque | ~20% | 30-60 jours | Modérée | Campagne réactivation, enquête satisfaction |
| 👋 Perdus | ~20% | > 90 jours | Faible | Email last-chance -20%, sinon abandon |

**ROI estimé** : une campagne de réactivation ciblée sur les « À Risque » peut récupérer **15-25%** du CA menacé.